In [1]:
# put this at the top of every notebook
import plotly.io as pio
pio.renderers.default = "notebook_connected"
#pio.renderers.default = "browser"   # always opens in browser, no nbformat needed


In [2]:
# Cell 1 — setup
from yfinance_api3.classes.stock_client import StockClient
from yfinance_api3.classes.quant_analytics import QuantAnalytics
from yfinance_api3.classes.options import OptionsAnalyzer
import yfinance_api3.modules.plots as plots

client = StockClient()
quant  = QuantAnalytics(client)
opt    = OptionsAnalyzer(client, "SPY")

In [3]:
# Cell 2 — expiries
print(opt.expiries())


['2026-05-01', '2026-05-04', '2026-05-05', '2026-05-06', '2026-05-07', '2026-05-08', '2026-05-15', '2026-05-22', '2026-05-29', '2026-06-05', '2026-06-18', '2026-06-30', '2026-07-17', '2026-07-31', '2026-08-21', '2026-08-31', '2026-09-18', '2026-09-30', '2026-12-18', '2026-12-31', '2027-01-15', '2027-03-19', '2027-03-31', '2027-06-17', '2027-09-17', '2027-12-17', '2028-01-21', '2028-06-16', '2028-12-15']


In [4]:
# Cell 3 — front month chain
expiry = opt.nearest_expiry(0)
print(f"Front month: {expiry}")
df = opt.chain(expiry)
print(df.head(20))

Front month: 2026-05-01
        expiry  type  strike     bid     ask  last_price  volume  \
0   2026-05-01  call   400.0  324.08  324.19      317.13    11.0   
1   2026-05-01   put   400.0    0.00    0.01        0.01   136.0   
2   2026-05-01   put   405.0    0.00    0.01        0.03   110.0   
3   2026-05-01  call   405.0  319.32  319.42      305.21   296.0   
4   2026-05-01  call   410.0  312.82  315.60      304.07     2.0   
5   2026-05-01   put   410.0    0.00    0.01        0.01   200.0   
6   2026-05-01   put   415.0    0.00    0.01        0.01    11.0   
7   2026-05-01  call   415.0  307.72  310.54      299.14    12.0   
8   2026-05-01  call   420.0  302.80  305.61      294.16    10.0   
9   2026-05-01   put   420.0    0.00    0.01        0.01    10.0   
10  2026-05-01   put   425.0    0.00    0.01        0.01    10.0   
11  2026-05-01   put   430.0    0.00    0.01        0.01   200.0   
12  2026-05-01   put   435.0    0.00    0.01        0.01    10.0   
13  2026-05-01  call   4

In [5]:
# Cell 4 — summary
print(opt.summary(expiry))

{'symbol': 'SPY', 'expiry': '2026-05-01', 'days_to_expiry': 0, 'n_strikes': 213, 'total_call_oi': 379900, 'total_put_oi': 1068975, 'total_call_vol': 861446, 'total_put_vol': 780903, 'put_call_ratio_oi': 2.814, 'put_call_ratio_vol': 0.907, 'max_pain_strike': 708.0, 'max_oi_call_strike': np.float64(708.0), 'max_oi_put_strike': np.float64(510.0), 'avg_call_iv': 1.2764, 'avg_put_iv': 0.8414, 'iv_skew': -0.435}


In [6]:
# Cell 5 — max pain (knockout price)
print(f"Max pain: ${opt.max_pain(expiry):,.2f}")

Max pain: $708.00


In [7]:
# Cell 6 — put/call ratio
print(opt.put_call_ratio())

        expiry  days_to_expiry  call_oi   put_oi  put_call_ratio
0   2026-05-01               0   379900  1068975           2.814
1   2026-05-04               3    86824   151945           1.750
2   2026-05-05               4    17345    32205           1.857
3   2026-05-06               5    11723    19338           1.650
4   2026-05-07               6     9166    16726           1.825
5   2026-05-08               7   227236   563203           2.478
6   2026-05-15              14   726062  2805970           3.865
7   2026-05-22              21    92129   533164           5.787
8   2026-05-29              28   229530  1290645           5.623
9   2026-06-05              35    18012    32269           1.792
10  2026-06-18              48   831692  1941389           2.334
11  2026-06-30              60   158802   301518           1.899
12  2026-07-17              77   174294   299620           1.719
13  2026-07-31              91    82616   139049           1.683
14  2026-08-21           

In [8]:
plots.options_chain(opt, opt.nearest_expiry(1)).show()

In [9]:
plots.options_oi_profile(opt, opt.nearest_expiry(1)).show()

In [10]:
plots.options_put_call(opt).show()

In [11]:
plots.options_surface(opt, max_expiries=6).show()


In [12]:
# front 12 expiries (~1 year for weeklies, or full year for monthlies)
plots.options_max_pain(opt, max_expiries=12).show()



In [13]:
# more expiries
plots.options_max_pain(opt, max_expiries=24).show()

In [14]:
plots.options_gex(opt, max_expiries=12).show()


In [15]:
plots.options_unusual(opt, vol_oi_threshold=3.0, min_volume=100).show()

In [16]:
df = opt.greeks(opt.nearest_expiry(0))
print("Greeks columns:", df.columns.tolist())

Greeks columns: ['expiry', 'type', 'strike', 'bid', 'ask', 'last_price', 'volume', 'open_interest', 'implied_volatility', 'in_the_money', 'contract', 'delta', 'gamma', 'theta', 'vega', 'rho', 'theoretical_price', 'greek_source']


In [17]:
#opt = OptionsAnalyzer(client, "SPY")

# Greeks

print(df[["type","strike","delta","gamma","theta","vega","rho","greek_source"]].head(10))


   type  strike   delta     gamma   theta    vega     rho   greek_source
0  call   400.0  1.0000  0.000000 -0.0548  0.0000  0.0110  black_scholes
1   put   400.0 -0.0002  0.000006 -0.0429  0.0003 -0.0000  black_scholes
2   put   405.0 -0.0001  0.000005 -0.0333  0.0002 -0.0000  black_scholes
3  call   405.0  1.0000  0.000000 -0.0555  0.0000  0.0111  black_scholes
4  call   410.0  1.0000  0.000000 -0.0562  0.0000  0.0112  black_scholes
5   put   410.0 -0.0002  0.000006 -0.0435  0.0003 -0.0000  black_scholes
6   put   415.0 -0.0002  0.000005 -0.0332  0.0002 -0.0000  black_scholes
7  call   415.0  1.0000  0.000000 -0.0568  0.0000  0.0114  black_scholes
8  call   420.0  1.0000  0.000000 -0.0575  0.0000  0.0115  black_scholes
9   put   420.0 -0.0002  0.000007 -0.0437  0.0003 -0.0000  black_scholes


In [18]:
# GEX
strike_gex = opt.gex_by_strike()
by_exp     = opt.gex_by_expiry()
total      = opt.gex_total()

flip     = total.get("flip_strike")
flip_str = f"${flip:,.2f}" if flip is not None else "none detected"
print(f"\nRegime:          {total['regime_label']}")
print(f"Total GEX:       ${total['total_net_gex']/1e6:.1f}M")
print(f"GEX flip:        {flip_str}")
print(f"Dominant expiry: {total.get('dominant_expiry', '—')}")
# Unusual activity
unusual = opt.unusual_activity(vol_oi_threshold=3.0, min_volume=100)
print(f"\nUnusual activity — {len(unusual)} strikes flagged")
if not unusual.empty:
    print(unusual[["type","strike","expiry","volume",
                   "open_interest","vol_oi_ratio","signal"]].head(10))


Regime:          Positive GEX — market makers are net long gamma. They sell rallies and buy dips → price-stabilising, expect lower realised volatility.
Total GEX:       $4326.3M
GEX flip:        $643.00
Dominant expiry: 2026-05-01

Unusual activity — 124 strikes flagged
   type  strike      expiry    volume  open_interest  vol_oi_ratio     signal
0   put   726.0  2026-05-01   15613.0             71        219.90  🚨 Unusual
1   put   722.0  2026-05-01  119787.0            894        133.99  🚨 Unusual
2   put   725.0  2026-05-01   19057.0            191         99.77  🚨 Unusual
3   put   723.0  2026-05-01   74731.0           1564         47.78  🚨 Unusual
4   put   721.0  2026-05-01   97880.0           2505         39.07  🚨 Unusual
5  call   723.0  2026-05-01  151152.0           6001         25.19  🚨 Unusual
6   put   724.0  2026-05-01   58005.0           2809         20.65  🚨 Unusual
7   put   720.0  2026-05-01   84102.0           4363         19.28  🚨 Unusual
8  call   724.0  2026-05-0

In [19]:
total = opt.gex_total()
print(total.keys())

dict_keys(['symbol', 'total_call_gex', 'total_put_gex', 'total_net_gex', 'regime', 'regime_label', 'flip_strike', 'dominant_expiry'])


In [20]:
by_exp = opt.gex_by_expiry()
print(by_exp.columns.tolist())
print(by_exp.head(2))

['expiry', 'days_to_expiry', 'call_gex', 'put_gex', 'net_gex', 'abs_gex']
       expiry  days_to_expiry      call_gex       put_gex       net_gex  \
0  2026-05-01               0  4.885586e+09 -2.155291e+09  2.730295e+09   
1  2026-05-04               3  1.267537e+09 -9.796968e+08  2.878407e+08   

        abs_gex  
0  3.197722e+09  
1  7.783674e+08  


In [21]:
from yfinance_api3.classes.pricing import (
    ContractType, Space, ExerciseType
)
from yfinance_api3.classes.positions_book import (
    PositionsBook, PortfolioBook, WatchList
)

# ── Build INTC book ───────────────────────────────────────────────────────
book = PositionsBook(
    symbol         = "INTC",
    spot           = 93.22,
    vol            = 0.61,           # 61% hist vol
    risk_free_rate = 4.50,           # 4.50%
    div_yield      = 0.0,
    contract_type  = ContractType.STOCK,
    lot_size       = 100,
)

# add legs from your sheet (negative lots = short)
book.add_option("call", "2026-12-18", 85, -60, 18.11, 0.61, "equity")
book.add_option("call", "2026-12-18", 90, -40, 21.50, 0.61, "equity")
book.add_option("call", "2026-01-30", 40,   0,  1.80, 0.54, "equity")
book.add_option("call", "2026-06-18", 40,   0,  4.50, 0.61, "equity")
book.add_option("call", "2026-06-18", 50,   0,  7.30, 0.61, "equity")
book.add_option("call", "2026-01-30", 37,   0, 10.40, 0.52, "equity")

# underlying position
book.add_underlying(lots=10000, entry_price=30.0, direction="long")

# ── Greeks table ──────────────────────────────────────────────────────────
print(book.greeks_table(days_ahead=0).to_string())

# ── Summary ───────────────────────────────────────────────────────────────
s = book.summary(days_ahead=0)
for k, v in s.items():
    print(f"  {k:20s}: {v}")

# ── +30 days simulation ───────────────────────────────────────────────────
print("\n--- +30 days ---")
s30 = book.summary(days_ahead=30)
for k, v in s30.items():
    print(f"  {k:20s}: {v}")

# ── Save ─────────────────────────────────────────────────────────────────
book.save("positions/INTC.json")
print("\nSaved to positions/INTC.json")

# ── Portfolio ─────────────────────────────────────────────────────────────
port = PortfolioBook(name="My portfolio")
port.add_book(book, path="positions/INTC.json")
print(port.summary())
port.save("positions/portfolio.json")

# ── WatchList ─────────────────────────────────────────────────────────────
wl = WatchList()
wl.add("INTC", notes="short calls position")
wl.add("SPY",  notes="hedge")
wl.save("positions/watchlist.json")

     leg_id         type      expiry strike     iv          model exercise   lots lot_size price_paid model_price      value        pnl     delta  gamma     vega   theta      rho
0  fe4d9ad0         Call  2026-12-18   85.0  61.0%  Black-Scholes        E    -60      100      18.11     22.6668 -136000.62  -27340.62  -4130.89 -46.89 -1573.12  238.42 -1576.37
1  eb2dee70         Call  2026-12-18   90.0  61.0%  Black-Scholes        E    -40      100       21.5       20.34  -81359.84    4640.16  -2582.84 -32.90 -1103.58  165.36 -1008.88
2  5e77083a         Call  2026-01-30   40.0  54.0%  Black-Scholes        E      0      100        1.8       53.22       0.00       0.00      0.00   0.00     0.00    0.00     0.00
3  89c77cb4         Call  2026-06-18   40.0  61.0%  Black-Scholes        E      0      100        4.5     53.4562       0.00       0.00      0.00   0.00     0.00   -0.00     0.00
4  bd5053a6         Call  2026-06-18   50.0  61.0%  Black-Scholes        E      0      100        7.3    

In [22]:
from yfinance_api3.classes.pricing import Space, ContractType
from yfinance_api3.classes.positions_book import PositionsBook, PortfolioBook

# rebuild INTC book
book = PositionsBook(
    symbol="INTC", spot=93.22, vol=0.61,
    risk_free_rate=4.50, div_yield=0.0,
)
book.add_option("call", "2026-12-18", 85, -60, 18.11, 0.61)
book.add_option("call", "2026-12-18", 90, -40, 21.50, 0.61)
book.add_underlying(lots=10000, entry_price=30.0, direction="long")

# single ticker dashboard — replicates your sheet
#plots.positions_book(book, days_ahead=0).show()
#plots.positions_book(book, days_ahead=30).show()   # +30 days simulation

plots.positions_book(book, days_ahead=30).show()   # chart + summary


In [23]:
plots.positions_legs(book, days_ahead=30).show()   # legs table



In [24]:
# portfolio summary
port = PortfolioBook(name="My portfolio")
port.add_book(book)
plots.portfolio_summary(port, days_ahead=0).show()

In [25]:
# Debug GEX
gex = opt.gex_by_strike()
print("GEX sample:")
print(gex.head(5))
print(f"\nMax abs net_gex: {gex['net_gex'].abs().max():.2f}")
print(f"Non-zero rows: {(gex['net_gex'] != 0).sum()} / {len(gex)}")

# Check gamma values
df = opt.greeks(opt.nearest_expiry(0))
print(f"\nGamma stats:")
print(df["gamma"].describe())
print(f"greek_source counts: {df['greek_source'].value_counts().to_dict()}")

GEX sample:
   strike  call_gex      put_gex      net_gex  cumulative_gex
0   400.0       0.0 -7928.837834 -7928.837834    -7928.837834
1   405.0       0.0  -321.227620  -321.227620    -8250.065454
2   410.0       0.0 -1977.508566 -1977.508566   -10227.574020
3   415.0       0.0 -1185.669426 -1185.669426   -11413.243446
4   420.0       0.0 -6200.998865 -6200.998865   -17614.242310

Max abs net_gex: 719969159.61
Non-zero rows: 213 / 213

Gamma stats:
count    382.000000
mean       0.006431
std        0.022464
min        0.000000
25%        0.000062
50%        0.000679
75%        0.003044
max        0.299146
Name: gamma, dtype: float64
greek_source counts: {'black_scholes': 382}


In [26]:
df = opt.greeks(opt.nearest_expiry(0))
print(df[["type","strike","implied_volatility","gamma","greek_source"]].head(10))
print(f"\nIV stats: min={df['implied_volatility'].min()}, max={df['implied_volatility'].max()}")
print(f"Zero IV rows: {(df['implied_volatility'] == 0).sum()} / {len(df)}")

   type  strike  implied_volatility     gamma   greek_source
0  call   400.0            5.656253  0.000000  black_scholes
1   put   400.0            3.250002  0.000006  black_scholes
2   put   405.0            3.125002  0.000005  black_scholes
3  call   405.0            5.696292  0.000000  black_scholes
4  call   410.0            5.493167  0.000000  black_scholes
5   put   410.0            3.125002  0.000006  black_scholes
6   put   415.0            3.000002  0.000005  black_scholes
7  call   415.0            5.338871  0.000000  black_scholes
8  call   420.0            5.282230  0.000000  black_scholes
9   put   420.0            3.000002  0.000007  black_scholes

IV stats: min=1.0000000000000003e-05, max=5.6962919421386715
Zero IV rows: 0 / 382
